# Global Dataset Preprocessing & Cleaning

## Objective
Menyiapkan dataset global agar:
- Bersih (clean)
- Konsisten
- Memiliki struktur yang sama dengan dataset local
- Siap digunakan untuk EDA dan comparison

# **Load Dataset**

In [ ]:
import pandas as pd
import numpy as np

df_global = pd.read_csv('data/job_dataset.csv')
print(df_global.shape)
df_global.head()

(1068, 7)


,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords
0,NET-F-001,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,Assist in coding and debugging applications; L...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
1,NET-F-002,.NET Developer,Fresher,0-1,C#; .NET Framework basics; ASP.NET; Razor; HTM...,Write simple C# programs under guidance; Suppo...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
2,NET-F-003,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Core; ASP.NET MVC; HTM...,Contribute to development of small modules; As...,.NET; C#; ASP.NET MVC; SQL Server; Entity Fram...
3,NET-F-004,.NET Developer,Fresher,0-1,C#; .NET Framework; ASP.NET basics; SQL Server...,Support in software design documentation; Assi...,.NET; C#; SQL Server; Entity Framework; ASP.NET
4,NET-F-005,.NET Developer,Fresher,0-1,C#; ASP.NET; MVC; Entity Framework basics; SQL...,Learn to design and build ASP.NET applications...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...


# **Handle Missing Values**

In [ ]:
# Identifikasi kolom yang memiliki null
print("Jumlah Null per Kolom:")
print(df_global.isnull().sum())

df_global = df_global.dropna(subset=['title', 'skills'])

df_global = df_global.reset_index(drop=True)

Jumlah Null per Kolom:
JobID                0
title                0
ExperienceLevel      0
YearsOfExperience    0
skills               0
Responsibilities     0
Keywords             0
is_tech              0
dtype: int64


# **Filter Data (Tech Only)**

In [ ]:
tech_keywords = ['data', 'developer', 'engineer', 'ai', 'software', 'cloud', 'analyst', 'programmer', 'intelligence']
pattern = '|'.join(tech_keywords)

# Buat kolom is_tech
df_global['is_tech'] = df_global['title'].str.contains(pattern, case=False, na=False)

# Filter hanya yang True
df_global = df_global[df_global['is_tech'] == True].reset_index(drop=True)

print(f"Data setelah difilter (Tech Only): {len(df_global)}")

Data setelah difilter (Tech Only): 718


# **Standardisasi Kolom**

In [ ]:
if 'company_name' not in df_global.columns:
    df_global['company_name'] = "Global Company"

if 'location_city' not in df_global.columns:
    df_global['location_city'] = "Remote/International"

cols_to_keep = ['title', 'company_name', 'skills', 'location_city']
df_global = df_global[cols_to_keep]

# **Cleaning & Normalisasi Skills**


In [ ]:
def clean_skills_step(text):
    if pd.isna(text): return ""
    cleaned = str(text).lower().replace('[', '').replace(']', '').replace("'", "").replace(";", ",")
    return cleaned

df_global['skills_cleaned'] = df_global['skills'].apply(clean_skills_step)

# **Feature Engineering**


In [ ]:
df_global['skills_count'] = df_global['skills_cleaned'].apply(lambda x: len(x.split(',')) if x != "" else 0)

def map_role_category(title):
    t = str(title).lower()
    if 'data' in t: return 'Data'
    if 'frontend' in t or 'backend' in t or 'fullstack' in t or 'software' in t: return 'Software Engineering'
    if 'ai' in t or 'machine learning' in t: return 'AI/ML'
    return 'Other Tech'

df_global['role'] = df_global['title'].apply(map_role_category)

# **Final Check & Save**


In [ ]:
print("Info Dataset Global Terakhir:")
print(df_global.info())

df_global.to_csv('global_jobs_cleaned.csv', index=False)
print("--- PROSES SELESAI: File 'global_jobs_cleaned.csv' telah dibuat ---")

Info Dataset Global Terakhir:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 718 entries, 0 to 717
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   title           718 non-null    object
 1   company_name    718 non-null    object
 2   skills          718 non-null    object
 3   location_city   718 non-null    object
 4   skills_cleaned  718 non-null    object
 5   skills_count    718 non-null    int64 
 6   role            718 non-null    object
dtypes: int64(1), object(6)
memory usage: 39.4+ KB
None
--- PROSES SELESAI: File 'global_jobs_cleaned.csv' telah dibuat ---
